## 0. Overview

This notebook will go through x main validation methods to check whether this proposed framework follows the following evaluation criteria:

1. Distribution Consistency
Whether the human mobility representation distribution is consistent as the demographic attributes' logical mobility distribution.

2. Behavioural Plausibility
To validate that under the constraint handlings, the generated mobility representation follows the logical constraints. This method will take several representative profiles.

3. Model Sensitivity Analysis
The opportunity layer adopted the log transformation, this method will compare the results with and without log transformation.

### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

### 2. Set Up File Paths

In [2]:
import sys
from pathlib import Path

# Find repository root (../..)
repo_root = Path.cwd().parents[1]

# Add repository root to Python import path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

### 3. Load nts0611 json file

In [3]:
with open(repo_root / "data" / "nts" / "harmonised" / "nts0611a_trip_purpose_probability.json", "r") as f:
    nts0611 = json.load(f)

In [4]:
nts0611_df = pd.DataFrame(nts0611)

In [5]:
nts0611_df

,Year,Gender,Age,Commuting,Business,Education,Escort Education,Shopping,Other Escort,Personal Business,Visiting Friends at Private Home,Visiting Friends Elsewhere,Entertainment or Public Activity,Sport: Participate,Holiday: Base,Day Trip,Just Walk
0,2024,Female,0-4,0.003616,0.000701,0.327535,0.044123,0.031882,0.142315,0.075573,0.106521,0.043625,0.078185,0.035980,0.015151,0.041132,0.053662
1,2024,Female,5-11,0.003616,0.000701,0.327535,0.044123,0.031882,0.142315,0.075573,0.106521,0.043625,0.078185,0.035980,0.015151,0.041132,0.053662
2,2024,Female,12-17,0.026672,0.001690,0.304515,0.038551,0.056038,0.125005,0.074419,0.108590,0.051023,0.074196,0.031747,0.015177,0.040354,0.052022
3,2024,Female,18-22,0.180319,0.015662,0.117592,0.019347,0.189261,0.036292,0.071061,0.109007,0.080139,0.057312,0.007461,0.017746,0.035592,0.063210
4,2024,Female,23-34,0.187671,0.030108,0.019513,0.088612,0.184447,0.069928,0.074421,0.086093,0.055916,0.059392,0.006704,0.017174,0.035667,0.084355
5,2024,Female,35-49,0.138959,0.030173,0.002271,0.170156,0.180556,0.100192,0.075522,0.064854,0.043113,0.053496,0.009181,0.011911,0.036452,0.083165
6,2024,Female,50-64,0.131147,0.032095,0.000690,0.039488,0.222626,0.066860,0.105210,0.088050,0.052823,0.062339,0.008032,0.013860,0.046023,0.130756
7,2024,Female,65-74,0.045730,0.013056,0.000397,0.021163,0.285039,0.047241,0.141740,0.109224,0.070948,0.075429,0.008485,0.013501,0.049347,0.118700
8,2024,Female,75+,0.011872,0.003727,0.000087,0.012801,0.324676,0.032908,0.159272,0.113154,0.082330,0.078310,0.007110,0.012737,0.047206,0.113809
9,2024,Male,0-4,0.003106,0.000525,0.333408,0.047855,0.023888,0.139715,0.068052,0.093075,0.037175,0.067786,0.065606,0.012175,0.048738,0.058896


In [7]:
# AI Collaboration Record
# Tool: ChatGPT
# Purpose: Validate that trip-purpose probabilities sum to 1
# for every demographic group in the processed NTS0611 dataset.

profile_columns = ["Year", "Gender", "Age"]

purpose_columns = [
    column for column in nts0611_df.columns
    if column not in profile_columns
]

# Confirm that all selected probability columns are numeric
probability_data = nts0611_df[purpose_columns].apply(
    pd.to_numeric,
    errors="raise"
)

# Sum all trip-purpose probabilities horizontally for each profile
probability_sums = probability_data.sum(axis=1)

# Check whether each distribution sums to 1
valid_sums = np.isclose(
    probability_sums,
    1.0,
    atol=1e-8
)

# Save any failed profiles for inspection
invalid_rows = nts0611_df.loc[
    ~valid_sums,
    profile_columns
].copy()

invalid_rows["Probability Sum"] = probability_sums[~valid_sums]

print(f"Demographic groups checked: {len(nts0611_df)}")
print(f"Passed: {valid_sums.sum()}")
print(f"Failed: {(~valid_sums).sum()}")

assert valid_sums.all(), (
    f"{(~valid_sums).sum()} demographic groups have "
    "trip-purpose probabilities that do not sum to 1."
)

print("PASS: All NTS0611 trip-purpose distributions sum to 1.")

Demographic groups checked: 18
Passed: 18
Failed: 0
PASS: All NTS0611 trip-purpose distributions sum to 1.
